# NPU域上板调试

NPU 域上板调试是算子开发验证阶段的核心环节，通过精准打印运行时数据与日志信息，可快速定位算子在 NPU 硬件执行过程中的逻辑错误、数据异常等问题。NPU 域上板数据打印功能主要包含DumpTensor与printf两类接口，NPU域上板数据打印功能包括DumpTensor、printf两种，其中DumpTensor用于打印指定Tensor的数据，printf主要用于打印标量和字符串信息。本文将从接口定位、使用规范、实操要点三个维度，系统介绍 NPU 域上板调试的核心方法。

本节学习大纲如下：

- NPU域上板打印接口使用说明
- NPU域上板调试实践

---
## 1. 环境准备

正式开始学习之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及使用Bisheng编译器，完成算子的开发及编译。

In [5]:
!mkdir -p Sources/07.03

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

---
## 2. NPU域上板打印接口使用说明

### 2.1 接口调用前提

两类接口均需**在算子 kernel 侧实现代码中**调用，且需确保代码已完成 NPU 硬件适配（如包含必要的头文件、遵循算子开发语法规范）。调用位置无严格限制，可根据调试需求，在算子初始化、核心计算逻辑、结果输出等关键节点插入接口调用语句。

### 2.2 DumpTensor接口功能说明与使用示例

DumpTensor 用于在算子执行过程中打印指定 Tensor 的内容，辅助调试与分析。接口支持附加自定义信息（desc 参数，仅支持 uint32_t 类型），例如行号或标识符，以便在多处调用时区分不同输出。

**函数原型**

- 无Tensor shape信息的打印
    ```
    template <typename T>
    __aicore__ inline void DumpTensor(const LocalTensor<T> &tensor, uint32_t desc, uint32_t dumpSize)
    template <typename T>
    __aicore__ inline void DumpTensor(const GlobalTensor<T>& tensor, uint32_t desc, uint32_t dumpSize)
    ```
- 带Tensor shape信息的打印
    ```
    template <typename T>
    __aicore__ inline void DumpTensor(const LocalTensor<T>& tensor, uint32_t desc, uint32_t dumpSize, const ShapeInfo& shapeInfo)
    template <typename T>
    __aicore__ inline void DumpTensor(const GlobalTensor<T>& tensor, uint32_t desc, uint32_t dumpSize, const ShapeInfo& shapeInfo)
    ```

**参数说明**

<table border="1" cellpadding="8" cellspacing="0" style="margin-left: 0; text-align: left;">
  <thead>
    <tr>
      <th>参数</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>tensor</code></td>
      <td>待打印的 Tensor，支持 LocalTensor（位于 Unified Buffer/L1/L0C）或 GlobalTensor（位于 Global Memory）。</td>
    </tr>
    <tr>
      <td><code>desc</code></td>
      <td>用户自定义附加信息（如行号），用于区分不同调用位置的输出。</td>
    </tr>
    <tr>
      <td><code>dumpSize</code></td>
      <td>需要打印的元素个数。</td>
    </tr>
    <tr>
      <td><code>shapeInfo</code></td>
      <td>传入Tensor的shape信息，可按照shape信息进行打印。当 <code>shapeInfo</code> 指定的尺寸大于 <code>dumpSize</code> 时，不足部分用 "-" 填充显示；若小于等于 <code>dumpSize</code>，则按实际形状打印，多余元素不显示。</td>
    </tr>
  </tbody>
</table>

**使用示例**

在算子kernel侧实现代码中调用DumpTensor接口，打印srcLocal张量的前dataLen个元素，并附加自定义编号5，便于在调试输出中快速定位该打印点：

```
AscendC::DumpTensor(srcLocal, 5, dataLen);
```


### 2.3 printf接口功能说明与使用示例

printf 接口提供格式化输出能力，其语法与标准 C/C++ printf 兼容，便于在算子核函数中打印变量值、调试信息等。接口支持两种调用形式：AscendC::printf 和 AscendC::PRINTF，二者功能完全相同。

**函数原型**

```
template <class... Args>
__aicore__ inline void printf(__gm__ const char* fmt, Args&&... args);

template <class... Args>
__aicore__ inline void PRINTF(__gm__ const char* fmt, Args&&... args);
```

**格式控制说明**

fmt 参数是格式控制字符串，由普通字符和转换说明组成。普通字符原样输出，转换说明以 % 开头，用于指定后续参数的类型和输出格式。在NPU域调试时，转换说明所支持的数据类型如下表所示：

<table border="1" cellpadding="8" cellspacing="0" style="margin-left: 0; text-align: left;">
  <thead>
    <tr>
      <th>转换说明</th>
      <th>描述</th>
      <th>NPU 域支持的数据类型</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>%d</code> / <code>%i</code></td>
      <td>输出十进制整数</td>
      <td>bool / int8_t / int16_t / int32_t / int64_t</td>
    </tr>
    <tr>
      <td><code>%u</code></td>
      <td>输出无符号十进制整数</td>
      <td>bool / uint8_t / uint16_t / uint32_t / uint64_t</td>
    </tr>
    <tr>
      <td><code>%x</code></td>
      <td>输出十六进制整数</td>
      <td>int8_t / int16_t / int32_t / int64_t / uint8_t / uint16_t / uint32_t / uint64_t</td>
    </tr>
    <tr>
      <td><code>%f</code></td>
      <td>输出实数（浮点数）</td>
      <td>float / half / bfloat16_t</td>
    </tr>
    <tr>
      <td><code>%s</code></td>
      <td>输出字符串</td>
      <td>字符串常量或指针</td>
    </tr>
    <tr>
      <td><code>%p</code></td>
      <td>输出指针地址</td>
      <td>指针类型</td>
    </tr>
  </tbody>
</table>

**使用示例**

在算子核函数中，可根据需要在关键位置插入 printf 语句，例如打印变量值、循环索引或调试标记。示例如下：

```
#include "kernel_operator.h"

// 整型打印：
AscendC::printf("fmt string %d\n", 0x123);
AscendC::PRINTF("fmt string %d\n", 0x123);

// 浮点型打印：
float a = 3.14;
AscendC::printf("fmt string %f\n", a);
AscendC::PRINTF("fmt string %f\n", a);

// 指针打印：
int *b;
AscendC::printf("TEST %p\n", b);
AscendC::PRINTF("TEST %p\n", b);
```


---

## 3. NPU域上板调试实践

本章将在[基于Add算子的核函数介绍](../02_AscendC_basic/02.04_introduction_to_kernel_functions_based_on_add_operator.ipynb)章节的Add算子工程基础上，演示如何通过添加 `DumpTensor` 和 `printf` 接口，在算子核函数的关键位置输出调试信息，帮助开发者直观了解NPU上的数据流动与计算状态。

### 3.1 添加DumpTensor接口

在Add算子核函数的`Compute`方法中，我们可以利用`DumpTensor`接口输出LocalTensor的内容，以便验证每个AI Core在首次计算时的输入与输出是否符合预期。以下示例展示了如何在`Compute`函数中插入`DumpTensor`语句，分别打印xLocal、yLocal和zLocal的前8个元素:

修改后的KernelAdd::Compute函数如下：
```
__aicore__ inline void KernelAdd::Compute(int32_t progress)
{
    // 从输入队列获取数据
    AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
    AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
    // 为计算结果分配内存
    AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();

    // 仅在第一次调用时打印输入和输出数据的前8个元素
    if (progress == 0) {
        AscendC::DumpTensor(xLocal, __LINE__, 8);
        AscendC::DumpTensor(yLocal, __LINE__, 8);
    }

    // 执行矢量加法计算
    AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);

    if (progress == 0) {
        AscendC::DumpTensor(zLocal, __LINE__, 8);
    }

    // 将计算结果放入输出队列，通知CopyOut任务
    outQueueZ.EnQue<float>(zLocal);
    // 释放输入数据内存（重要：避免内存泄漏）
    inQueueX.FreeTensor(xLocal);
    inQueueY.FreeTensor(yLocal);
}
```

### 3.2 添加printf接口

在Add算子核函数的`Init`方法中，我们可以利用`printf`接口输出关键的切分参数，以便验证多核并行场景下的数据划分是否符合预期。以下示例展示了如何在KernelAdd::Init函数中插入`printf`语句，分别打印数据总长度、每个核心的数据切块数以及参与计算的AI Core总数。

修改后的KernelAdd::Init函数如下：

```
__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
{
    // 计算每个核处理的数据长度（多核并行）
     this->blockLength = totalLength / AscendC::GetBlockNum();     // length computed of each core
    // 记录单核内数据切块参数
     this->tileNum = tileNum;                                      // split data into 8 tiles for each core
     this->tileLength = this->blockLength / tileNum / BUFFER_NUM;  // separate to 2 parts, due to double buffer
    
    // 打印关键参数，验证切分逻辑
    AscendC::printf("totalLength: %d, tileNum: %d, blockNum: %lu\n", 
                    totalLength, tileNum, AscendC::GetBlockNum());

    // 设置每个核的Global Memory起始地址（关键的多核切分逻辑）
     xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
     zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
    // 为队列分配内存（单核内流水并行的基础）
     pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
     pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
}
```

## 3.3 执行验证

将如上添加打印信息的改动写入Sources/07.03/add_custom.asc文件中：

In [ ]:
%%writefile Sources/07.03/add_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        // 打印关键参数，验证切分逻辑
        AscendC::printf("totalLength: %d, tileNum: %d, blockNum: %lu\n", 
                        totalLength, tileNum, AscendC::GetBlockNum());
        xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
        AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();
        // 仅在第一次调用时打印输入和输出数据的前8个元素
        if (progress == 0) {
            AscendC::DumpTensor(xLocal, __LINE__, 8);
            AscendC::DumpTensor(yLocal, __LINE__, 8);
        }
        
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        
        if (progress == 0) {
            AscendC::DumpTensor(zLocal, __LINE__, 8);
        }
        outQueueZ.EnQue<float>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    add_custom<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);

    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}

执行以下命令进行编译并查看打印结果：

In [ ]:
%%writefile Sources/07.03/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
# find_package(ASC)是CMake中用于查找和配置Ascend C编译工具链的命令
find_package(ASC REQUIRED)
# 指定项目支持的语言包括ASC和CXX，ASC表示支持使用毕昇编译器对Ascend C编程语言进行编译
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_custom.asc
)
# 通过编译选项设置NPU架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

In [ ]:
!cd Sources/07.03 && \
mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/07.03/build/ && \
cmake .. && \
make && \
./demo

## 课后实践

实践要求：请修改如下 add_custom.asc 源码文件，在核函数的 Compute 函数中调用 DumpTensor 接口添加打印语句。具体要求如下：

- 仅在逻辑 ID 为 2 的 AI Core 最后一次执行 Compute 函数时触发打印；

- 打印 zLocal 张量的前 8 个元素；

- 使用带 Tensor shape 的 DumpTensor 接口，并按 shape 为 (2, 8) 的格式显示。

完成修改后，编译运行程序，观察控制台输出的日志信息，验证程序执行流程是否符合预期。


### 1. 添加打印信息

在如下代码框中修改添加打印信息：

In [ ]:
%%writefile Sources/07.03/add_custom.asc

#include <cstdint>
#include <iostream>
#include <vector>
#include <algorithm>
#include <iterator>
#include "acl/acl.h"
#include "kernel_operator.h"

constexpr uint32_t BUFFER_NUM = 2; // tensor num for each queue

struct AddCustomTilingData
{
    uint32_t totalLength;
    uint32_t tileNum;
};

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z, uint32_t totalLength, uint32_t tileNum)
    {
        this->blockLength = totalLength / AscendC::GetBlockNum();
        this->tileNum = tileNum;
        this->tileLength = this->blockLength / tileNum / BUFFER_NUM;
        xGm.SetGlobalBuffer((__gm__ float *)x + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        yGm.SetGlobalBuffer((__gm__ float *)y + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        zGm.SetGlobalBuffer((__gm__ float *)z + this->blockLength * AscendC::GetBlockIdx(), this->blockLength);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileLength * sizeof(float));
        pipe.InitBuffer(inQueueY, BUFFER_NUM, this->tileLength * sizeof(float));
        pipe.InitBuffer(outQueueZ, BUFFER_NUM, this->tileLength * sizeof(float));
    }
    __aicore__ inline void Process()
    {
        int32_t loopCount = this->tileNum * BUFFER_NUM;
        for (int32_t i = 0; i < loopCount; i++) {
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.AllocTensor<float>();
        AscendC::LocalTensor<float> yLocal = inQueueY.AllocTensor<float>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileLength], this->tileLength);
        AscendC::DataCopy(yLocal, yGm[progress * this->tileLength], this->tileLength);
        inQueueX.EnQue(xLocal);
        inQueueY.EnQue(yLocal);
    }
    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<float> xLocal = inQueueX.DeQue<float>();
        AscendC::LocalTensor<float> yLocal = inQueueY.DeQue<float>();
        AscendC::LocalTensor<float> zLocal = outQueueZ.AllocTensor<float>();        
        AscendC::Add(zLocal, xLocal, yLocal, this->tileLength);
        // 请添加打印信息...
        
        outQueueZ.EnQue<float>(zLocal);
        inQueueX.FreeTensor(xLocal);
        inQueueY.FreeTensor(yLocal);
    }
    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<float> zLocal = outQueueZ.DeQue<float>();
        AscendC::DataCopy(zGm[progress * this->tileLength], zLocal, this->tileLength);
        outQueueZ.FreeTensor(zLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::TPosition::VECIN, BUFFER_NUM> inQueueX, inQueueY;
    AscendC::TQue<AscendC::TPosition::VECOUT, BUFFER_NUM> outQueueZ;
    AscendC::GlobalTensor<float> xGm;
    AscendC::GlobalTensor<float> yGm;
    AscendC::GlobalTensor<float> zGm;
    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, AddCustomTilingData tiling)
{
    KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_AIV_ONLY);
    KernelAdd op;
    op.Init(x, y, z, tiling.totalLength, tiling.tileNum);
    op.Process();
}

std::vector<float> kernel_add(std::vector<float> &x, std::vector<float> &y)
{
    constexpr uint32_t blockDim = 8;
    uint32_t totalLength = x.size();
    size_t totalByteSize = totalLength * sizeof(float);
    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    AddCustomTilingData tiling = {/*totalLength:*/totalLength, /*tileNum:*/8};
    uint8_t *xHost = reinterpret_cast<uint8_t *>(x.data());
    uint8_t *yHost = reinterpret_cast<uint8_t *>(y.data());
    uint8_t *zHost = nullptr;
    uint8_t *xDevice = nullptr;
    uint8_t *yDevice = nullptr;
    uint8_t *zDevice = nullptr;

    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    aclrtMallocHost((void **)(&zHost), totalByteSize);
    aclrtMalloc((void **)&xDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&yDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void **)&zDevice, totalByteSize, ACL_MEM_MALLOC_HUGE_FIRST);

    aclrtMemcpy(xDevice, totalByteSize, xHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(yDevice, totalByteSize, yHost, totalByteSize, ACL_MEMCPY_HOST_TO_DEVICE);

    add_custom<<<blockDim, nullptr, stream>>>(xDevice, yDevice, zDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(zHost, totalByteSize, zDevice, totalByteSize, ACL_MEMCPY_DEVICE_TO_HOST);
    std::vector<float> z((float *)zHost, (float *)(zHost + totalByteSize));

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFree(zDevice);
    aclrtFreeHost(zHost);

    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();

    return z;
}

uint32_t VerifyResult(std::vector<float> &output, std::vector<float> &golden)
{
    auto printTensor = [](std::vector<float> &tensor, const char *name) {
        constexpr size_t maxPrintSize = 20;
        std::cout << name << ": ";
        std::copy(tensor.begin(), tensor.begin() + std::min(tensor.size(), maxPrintSize),
            std::ostream_iterator<float>(std::cout, " "));
        if (tensor.size() > maxPrintSize) {
            std::cout << "...";
        }
        std::cout << std::endl;
    };
    printTensor(output, "Output");
    printTensor(golden, "Golden");
    if (std::equal(golden.begin(), golden.end(), output.begin())) {
        std::cout << "[Success] Case accuracy is verification passed." << std::endl;
        return 0;
    } else {
        std::cout << "[Failed] Case accuracy is verification failed!" << std::endl;
        return 1;
    }
    return 0;
}

int32_t main(int32_t argc, char *argv[])
{
    constexpr uint32_t totalLength = 8 * 2048;
    constexpr float valueX = 1.2f;
    constexpr float valueY = 2.3f;
    std::vector<float> x(totalLength, valueX);
    std::vector<float> y(totalLength, valueY);

    std::vector<float> output = kernel_add(x, y);

    std::vector<float> golden(totalLength, valueX + valueY);
    return VerifyResult(output, golden);
}

### 2. 编译运行

直接执行以下命令编译并执行算子，查看调试信息，无需改动:

In [ ]:
%%writefile Sources/07.03/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)
# find_package(ASC)是CMake中用于查找和配置Ascend C编译工具链的命令
find_package(ASC REQUIRED)
# 指定项目支持的语言包括ASC和CXX，ASC表示支持使用毕昇编译器对Ascend C编程语言进行编译
project(kernel_samples LANGUAGES ASC CXX)

add_executable(demo
    add_custom.asc
)
# 通过编译选项设置NPU架构
target_compile_options(demo PRIVATE
    $<$<COMPILE_LANGUAGE:ASC>:--npu-arch=dav-2201>
)

In [ ]:
!cd Sources/07.03 && \
mkdir -p build
!export ASC_DIR=$ASCEND_HOME_PATH/aarch64-linux/tikcpp/ascendc_kernel_cmake/ && \
cd Sources/07.03/build/ && \
cmake .. && \
make && \
./demo

### 3. 验证结果

预期执行效果如下：

```
opType=add_custom, DumpHead: AIV-2, CoreType=AIV, block dim=8, total_block_num=8, block_remain_len=1048336, block_initial_space=1048576, rsv=0, magic=5aa5bccd
CANN Version: 8.5.0.alpha001, TimeStamp: 20251106224806120
DumpTensor: desc=0, addr=a00, data_type=float32, position=UB, dump_size=8
shape is [2, 8], dumpSize is 8, data is not enough.
[[3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000,3.500000],
[-,-,-,-,-,-,-,-]]
```

执行以下代码获取答案。

In [ ]:
!cat ./answer/07.03_answer.txt